# 06 — Q4: What should be taken into the spatial SPP1–CD44 analysis?

This notebook does **no spatial analysis**. It decides, from the single-cell evidence, which
(cell type × program) and which chemokine ligand → receptor axes are worth mapping against the
SPP1–CD44-high niches, and it checks — by gene detection only — whether each can actually be
measured in the spatial object.

Each candidate is scored on four things:

| criterion | source | why it matters spatially |
|---|---|---|
| **evidence tier** | `05` ledgers | only map what the single-cell data support |
| **cell-type specificity** | `01` per-cell level and contribution | the spatial label must carry it; a program made by everyone cannot be assigned to one label |
| **relation to SPP1 → CD44 in single cell** | per-sample LIANA (`rank_aggregate`, `lr_means`) vs program scores, car / rt / site partialled out | a reason to expect a spatial relationship at all |
| **spatial transferability** | per-label gene detection in the spatial object + a B-cell / other-label spillover control | `spp1Analysis` 01 §5 found B cells *more* Spp1-positive than macrophages; this is the same test for every gene here |

Plus an **audit of the R01 cytotoxicity score** (`GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY`),
the definition behind "cytotoxic-high" cells in the current spatial figure: it contains
`Fcgr4`, `Srgn`, `Lamp1` and `Bnip3`, which are not lymphocyte genes.

## 1. Config

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, time, logging
from pathlib import Path
import numpy as np, pandas as pd, h5py, anndata as ad, zarr
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
logging.disable(logging.WARNING)

pd.set_option("display.width", 300); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 500); pd.set_option("display.max_colwidth", 80)
_HPC = Path("/coh_labs/yunroseli/Jona/CAR-T"); _LOC = Path("/Users/janzules/Roselab/Spatial/CAR_T")
PROJ = _LOC if _LOC.exists() else _HPC
SC_REF = (PROJ/"data/sc-reference/annotation_v4.h5ad" if PROJ is _LOC
          else PROJ/"data/zarr/fullDataset/annotating_references/annotation_v4.h5ad")
ZARR = (PROJ/"data/zarrFiles/1_C2l_annotations_400" if PROJ is _LOC
        else PROJ/"data/zarr/fullDataset/processing_Zarr/1_C2l_annotated_CC")
OUT = PROJ/"results/mhc_cytotox"; FIG = OUT/"figures"
MIN_CELLS, N_FL, SEED, TOP_Q = 10, 5000, 0, 0.85
SP_CACHE = OUT/"06_spatial_detection.npz"

CAT = json.load(open(OUT/"programs.json")); PROGRAMS, AXES = CAT["programs"], CAT["axes"]
PB = pd.read_parquet(OUT/"pseudobulk_counts.parquet")
PBM = pd.read_csv(OUT/"pseudobulk_meta.csv").set_index("key").loc[PB.index]
L = np.log2(PB.div(PB.sum(1), axis=0) * 1e6 + 1)
design = pd.read_csv(OUT/"design_samples.csv").set_index("sample")
TREATED = design.index[design.arm != "NoTx"]
LED = pd.read_csv(OUT/"ledger_programs.csv"); CLED = pd.read_csv(OUT/"ledger_chemokines.csv")
AXT = pd.read_csv(OUT/"chemokine_axis_table_sc.csv")
Q1 = pd.read_csv(OUT/"q1_level_by_condition.csv"); CS = pd.read_csv(OUT/"q1_contribution_by_condition.csv")
SC_DET = pd.read_csv(OUT/"panel_gene_detection_by_celltype.csv", index_col=0)
cp = pd.read_parquet(OUT/"cell_panel.parquet")

TO_SPATIAL = {"IFN_stimulated_Mac": "M1_like_Mac", "M1_like_Mac": "M1_like_Mac", "SPP1_TAM": "M2_like_Mac",
              "M2_like_Mac": "M2_like_Mac", "Intermediate_Mac": "Intermediate_Mac", "Classical_Mono": "Classical_Mono",
              "cDC": "cDC", "Cancer_cell": "Cancer_cell", "NK": "NK", "CD8_T": "CD8_T", "cytotoxic_lymph": "CD8_T",
              "N1_like_Neu": "N1_like_Neu", "N2_like_Neu": "N2_like_Neu", "Fibroblast": "Fibroblast",
              "Endothelial": "Endothelial", "NKT": "NKT", "CD4_T": "CD4_T", "Treg": "Treg"}

## 2. The R01 cytotoxicity score — what does it measure in single cell?

Per cell: mean log-normalised expression of the 13 genes. Then: which cell types make up the
top 15% of scorers (the R01 figure thresholds near the top of the distribution), and which genes
drive the score inside those top cells.

In [2]:
R01 = [g for g in PROGRAMS["R01_granzyme_GO"]["genes"] if g in cp]
ln = np.log1p(cp[R01].div(cp.total_counts, axis=0) * 1e4)
cp["R01_score"] = ln.mean(1)
trc = cp[cp.arm != "NoTx"]
thr = trc.R01_score.quantile(TOP_Q)
top = trc[trc.R01_score >= thr]
comp_top = top.ct2.value_counts(normalize=True)
print(f"R01 set genes present: {R01}")
print(f"\nwho are the top {100*(1-TOP_Q):.0f}% R01 scorers (treated samples)?")
print((100 * comp_top).round(1).head(10).to_string())
print("\nmean R01 score per cell type (treated):")
print(trc.groupby("ct2").R01_score.mean().sort_values(ascending=False).round(3).head(10).to_string())
gshare = (ln.loc[top.index].groupby(top.ct2).mean().div(ln.loc[top.index].groupby(top.ct2).mean().sum(1), axis=0))
print("\nwithin the top scorers, share of the score from each gene (rows = cell type):")
print((100 * gshare.loc[comp_top.index[:6]]).round(0).to_string())

# a lymphocyte-restricted alternative: genes whose single-cell detection in CD8/NK is >= 4x their
# highest detection in any non-lymphoid type
LYM = ["CD8_T", "NK"]; NONLYM = [c for c in SC_DET.columns if c not in ["CD8_T", "NK", "NKT", "CD4_T", "Treg"]]
cand = list(dict.fromkeys(R01 + PROGRAMS["cyto_effector_noCcl5"]["genes"] + ["Ctsw", "Klrd1", "Klrk1", "Gzma"]))
cand = [g for g in cand if g in SC_DET.index]
spec = (SC_DET.loc[cand, LYM].max(1) / SC_DET.loc[cand, NONLYM].max(1).clip(lower=0.005))
ALT = spec[(spec >= 4) & (SC_DET.loc[cand, LYM].max(1) >= 0.2)].index.tolist()
print(f"\nlymphocyte-restricted cytotoxic alternative (sc detection CD8/NK >= 4x any non-lymphoid type): {ALT}")
cp["ALT_score"] = np.log1p(cp[ALT].div(cp.total_counts, axis=0) * 1e4).mean(1)
trc = cp[cp.arm != "NoTx"]
top_alt = trc[trc.ALT_score >= trc.ALT_score.quantile(TOP_Q)]
print("top 15% scorers on the alternative:")
print((100 * top_alt.ct2.value_counts(normalize=True)).round(1).head(8).to_string())

# "composition of the top 15%" is the wrong yardstick when lymphocytes are < 2% of cells: even a
# perfect lymphocyte score fills most of a top-15% slice with other cells. Two fairer measures:
# AUROC (lymphocyte vs everything else) and precision at k = the number of lymphocytes.
from sklearn.metrics import roc_auc_score
LYMPH_CT = ["CD8_T", "NK", "NKT", "CD4_T", "Treg"]


def discrim(score, is_lym):
    k = int(is_lym.sum()); top_k = np.argsort(-score)[:k]
    return roc_auc_score(is_lym, score), float(is_lym[top_k].mean()), k


sc_lym = trc.ct2.isin(LYMPH_CT).values
DISC = {}
for nm in ["R01_score", "ALT_score"]:
    DISC[("single cell", nm)] = discrim(trc[nm].values, sc_lym)
    print(f"single cell {nm:<10s} AUROC lymphocyte vs rest {DISC[('single cell', nm)][0]:.3f} | precision at "
          f"k={DISC[('single cell', nm)][2]:,} lymphocytes: {DISC[('single cell', nm)][1]:.2f} "
          f"(base rate {sc_lym.mean():.3f})")

R01 set genes present: ['Bnip3', 'Gzma', 'Gzmb', 'Gzmc', 'Gzmk', 'Lamp1', 'Nkg7', 'Prf1', 'Srgn', 'Ube4b']

who are the top 15% R01 scorers (treated samples)?
ct2
M2_like_Mac          44.7
Classical_Mono       22.6
M1_like_Mac          10.8
NK                    7.0
Intermediate_Mac      4.6
N1_like_Neu           2.4
CD8_T                 2.1
Nonclassical_Mono     1.8
N2_like_Neu           1.4
Cancer_cell           1.4

mean R01 score per cell type (treated):
ct2
NK                   1.153
CD8_T                0.712
CD4_T                0.630
Treg                 0.600
N2_like_Neu          0.572
M2_like_Mac          0.523
N1_like_Neu          0.497
Classical_Mono       0.458
M1_like_Mac          0.450
Nonclassical_Mono    0.429

within the top scorers, share of the score from each gene (rows = cell type):
                  Bnip3  Gzma  Gzmb  Gzmc  Gzmk  Lamp1  Nkg7  Prf1  Srgn  Ube4b
ct2                                                                            
M2_like_Mac        17.0

single cell R01_score  AUROC lymphocyte vs rest 0.822 | precision at k=1,679 lymphocytes: 0.57 (base rate 0.020)
single cell ALT_score  AUROC lymphocyte vs rest 0.953 | precision at k=1,679 lymphocytes: 0.83 (base rate 0.020)


## 3. Cell-type specificity of each candidate (from `01`)

`margin` = per-cell level in the group minus the highest level in any other lvl2 type
(log2 CPM; > 0 means the group leads). `share` = the group's share of the program's transcripts.

In [3]:
tq = Q1[~Q1.cond.str.startswith("NoTx")].groupby(["program", "group"])["level"].mean()
tc = CS[~CS.cond.str.startswith("NoTx")].groupby(["program", "group"])["share"].mean()
LVL2 = ["Cancer_cell", "M1_like_Mac", "M2_like_Mac", "Intermediate_Mac", "Classical_Mono", "Nonclassical_Mono",
        "N1_like_Neu", "N2_like_Neu", "cDC", "pDC", "CD8_T", "NK", "Fibroblast", "Endothelial"]


def specificity(prog, group):
    g2 = "M1_like_Mac" if group == "IFN_stimulated_Mac" else "M2_like_Mac" if group == "SPP1_TAM" else group
    if prog not in tq.index.get_level_values(0): return np.nan, np.nan
    lv = tq.loc[prog]
    own = lv.get(group, lv.get(g2, np.nan))
    others = lv.reindex([c for c in LVL2 if c != g2]).dropna()
    return own - others.max() if len(others) else np.nan, tc.get((prog, g2), np.nan)

## 4. SPP1 → CD44 in single cell, per sample (LIANA), and its relation to the programs

`liana.mt.rank_aggregate` per sample on lvl2 types with a **custom resource**: SPP1–CD44 plus
the chemokine pairs. The default `mouseconsensus` resource lacks several of them — printed
below, because a spatial LIANA run with that resource would silently never score those axes.

In [4]:
import liana as li
PAIRS = [("Spp1", "Cd44")] + [(l, r) for a in AXES.values() for l in a["ligands"] for r in a["receptors"]]
res_mc = li.resource.select_resource("mouseconsensus")
missing_mc = [f"{l}->{r}" for l, r in PAIRS if not ((res_mc.ligand == l) & (res_mc.receptor == r)).any()]
print(f"pairs NOT in liana mouseconsensus: {missing_mc}")
RES = pd.DataFrame(PAIRS, columns=["ligand", "receptor"])

LI_FILE = OUT/"06_sc_liana_per_sample.csv"
if LI_FILE.exists():
    LIR = pd.read_csv(LI_FILE)
else:
    with h5py.File(SC_REF, "r") as f:
        X = ad.io.read_elem(f["X"]); obs = ad.io.read_elem(f["obs"]); var = ad.io.read_elem(f["var"])
    full = ad.AnnData(X=X, obs=pd.DataFrame({"ct2": obs.cell_type_lvl2.astype(str).values,
                                             "sample": obs.file_name.astype(str).values}, index=obs.index),
                      var=pd.DataFrame(index=var.index.astype(str)))
    full = full[:, [g for g in set(RES.ligand) | set(RES.receptor) if g in full.var_names]].copy()
    out, t0 = [], time.time()
    for s in sorted(full.obs["sample"].unique()):
        a = full[(full.obs["sample"] == s).values & (full.obs.ct2 != "Erythrocyte").values].copy()
        li.mt.rank_aggregate(a, groupby="ct2", resource=RES, use_raw=False, expr_prop=0.1, min_cells=5,
                             n_perms=100, verbose=False, return_all_lrs=True)
        out.append(a.uns["liana_res"].assign(sample=s))
    LIR = pd.concat(out, ignore_index=True)
    LIR.to_csv(LI_FILE, index=False)
    print(f"LIANA on {LIR['sample'].nunique()} samples in {time.time()-t0:.0f}s")

spp = LIR[(LIR.ligand_complex == "Spp1") & (LIR.receptor_complex == "Cd44")]
SOURCES = ["Cancer_cell", "M2_like_Mac", "M1_like_Mac", "N1_like_Neu", "N2_like_Neu", "Classical_Mono"]
POT = (spp[spp.source.isin(SOURCES)].groupby(["sample", "target"]).lr_means.mean().unstack())
print("\nSPP1 -> CD44 potential (mean lr_means over sources), treated mean by receiver:")
print(POT.reindex(TREATED).mean().sort_values(ascending=False).round(3).head(10).to_string())


def freedman_lane(y, x, Zc, n_perm=N_FL, seed=SEED):
    g_ = np.random.default_rng(seed)
    ok = np.isfinite(y) & np.isfinite(x) & np.isfinite(Zc).all(1)
    y, x, Zc = y[ok], x[ok], Zc[ok]; n = len(y)
    if n < Zc.shape[1] + 4 or np.std(x) == 0: return np.nan, np.nan, n
    Xf = np.column_stack([np.ones(n), x, Zc]); Xr = np.column_stack([np.ones(n), Zc])
    def t(yy):
        b, *_ = np.linalg.lstsq(Xf, yy, rcond=None); e = yy - Xf @ b
        s2 = e @ e / (n - Xf.shape[1]); return b[1] / np.sqrt(s2 * np.linalg.pinv(Xf.T @ Xf)[1, 1])
    t0 = t(y); br, *_ = np.linalg.lstsq(Xr, y, rcond=None); fr = Xr @ br; e = y - fr
    null = np.array([t(fr + g_.permutation(e)) for _ in range(n_perm)])
    df = n - Xf.shape[1]
    return t0 / np.sqrt(t0 ** 2 + df), (1 + (np.abs(null) >= abs(t0)).sum()) / (1 + n_perm), n


def covars(smp):
    d = design.loc[smp]
    return np.column_stack([(d.cart == "PSCA").astype(float), d.RT.astype(float), (d.site == 1).astype(float)])


def score(group, genes_):
    d = PBM[(PBM.group == group) & (PBM.n_cells >= MIN_CELLS)]
    Lg = L.loc[d.index, [g for g in genes_ if g in L.columns]]; Lg = Lg.loc[:, Lg.mean() >= 1.0]
    tr = (d.arm != "NoTx").values
    Z = (Lg - Lg[tr].mean()) / Lg[tr].std().replace(0, np.nan)
    return pd.Series(Z.mean(1).values, index=d["sample"].values)


TESTS = [  # receiver of SPP1->CD44, (group, program or gene)
    ("CD8_T", "CD8_T", "cyto_effector_noCcl5"), ("CD8_T", "CD8_T", "exh_inhibitory_receptors"),
    ("NK", "NK", "cyto_effector_noCcl5"), ("NK", "cytotoxic_lymph", "cyto_effector_noCcl5"),
    ("M1_like_Mac", "IFN_stimulated_Mac", "MHC_II_core"), ("M1_like_Mac", "IFN_stimulated_Mac", "MHC_I_machinery"),
    ("M1_like_Mac", "IFN_stimulated_Mac", "IFNG_Hallmark"), ("M1_like_Mac", "IFN_stimulated_Mac", "CXCR3_ligands"),
    ("M2_like_Mac", "SPP1_TAM", "MHC_II_core"), ("M2_like_Mac", "SPP1_TAM", "CXCR3_ligands"),
    ("M2_like_Mac", "M2_like_Mac", "CCR2_ligands"), ("Classical_Mono", "Classical_Mono", "CXCR3_ligands"),
    ("Cancer_cell", "Cancer_cell", "MHC_I_machinery"), ("Cancer_cell", "Cancer_cell", "checkpoint_ligands"),
]
rows = []
for recv, grp, prog in TESTS:
    if recv not in POT: continue
    y = score(grp, PROGRAMS[prog]["genes"])
    smp = [s for s in TREATED if s in y.index and s in POT.index and np.isfinite(POT.loc[s, recv])]
    r, p, n = freedman_lane(y.loc[smp].values, POT.loc[smp, recv].values, covars(smp))
    # abundance of the receiving type as the second readout for the lymphoid receivers
    rows.append(dict(spp1_cd44_receiver=recv, group=grp, program=prog, n=n, partial_r=r, p_FL=p))
SPPX = pd.DataFrame(rows)
SPPX["q_BH"] = multipletests(SPPX.p_FL.fillna(1), method="fdr_bh")[1]
SPPX.to_csv(OUT/"06_spp1_cd44_vs_programs.csv", index=False)
print("\nper-sample SPP1->CD44 potential vs program score (treated, car/rt/site partialled out):")
print(SPPX.round(3).to_string(index=False))

pairs NOT in liana mouseconsensus: ['Cxcl9->Cxcr3', 'Ccl3->Ccr5', 'Ccl7->Ccr2', 'Cxcl3->Cxcr2', 'Ppbp->Cxcr2']

SPP1 -> CD44 potential (mean lr_means over sources), treated mean by receiver:
target
N1_like_Neu          1.737
Nonclassical_Mono    1.649
N2_like_Neu          1.618
Classical_Mono       1.594
M1_like_Mac          1.510
pDC                  1.463
Intermediate_Mac     1.459
M2_like_Mac          1.427
NKT                  1.390
CD4_T                1.046



per-sample SPP1->CD44 potential vs program score (treated, car/rt/site partialled out):
spp1_cd44_receiver              group                  program  n  partial_r  p_FL  q_BH
             CD8_T              CD8_T     cyto_effector_noCcl5 16     -0.511 0.080 0.200
             CD8_T              CD8_T exh_inhibitory_receptors 16     -0.675 0.011 0.104
                NK                 NK     cyto_effector_noCcl5 20     -0.162 0.543 0.585
                NK    cytotoxic_lymph     cyto_effector_noCcl5 23     -0.276 0.245 0.296
       M1_like_Mac IFN_stimulated_Mac              MHC_II_core 23     -0.371 0.100 0.200
       M1_like_Mac IFN_stimulated_Mac          MHC_I_machinery 23     -0.261 0.254 0.296
       M1_like_Mac IFN_stimulated_Mac            IFNG_Hallmark 23     -0.063 0.797 0.797
       M1_like_Mac IFN_stimulated_Mac            CXCR3_ligands 23      0.302 0.192 0.296
       M2_like_Mac           SPP1_TAM              MHC_II_core 23     -0.289 0.215 0.296
       M2_like_Mac   

## 5. Spatial transferability — detection per c2l label (read-only, one streaming pass)

For every candidate gene: the fraction of cells with a non-zero value in each `c2l_consolidated`
label, and per cell the R01 and alternative cytotoxic scores. **No spatial statistics** — only
whether a gene is detected where single cell says it should be, and not equally everywhere else.

In [5]:
SP_GENES = set(R01) | set(ALT) | {"Spp1", "Cd44"}
for k in ["MHC_I_machinery", "MHC_II_core", "ISG_core", "cyto_effector", "exh_inhibitory_receptors", "exh_TFs",
          "exh_progenitor", "exh_terminal", "checkpoint_ligands"]:
    SP_GENES |= set(PROGRAMS[k]["genes"])
SP_GENES |= {g for a in AXES.values() for g in a["ligands"] + a["receptors"]}
g = zarr.open_group(str(ZARR/"tables/segmentation_counts"), mode="r")
sp_genes = pd.Index(ad.io.read_elem(g["var"]).index.astype(str))
SP_GENES = [x for x in sorted(SP_GENES) if x in sp_genes]
so = ad.io.read_elem(g["obs"])[["tissue", "condition", "c2l_consolidated"]].astype(str)
labels = pd.Index(sorted(so.c2l_consolidated.unique())); lab_idx = labels.get_indexer(so.c2l_consolidated)

if SP_CACHE.exists():
    z_ = np.load(SP_CACHE, allow_pickle=True)
    DET_N, R01_SUM, ALT_SUM = z_["det"], z_["r01"], z_["alt"]
    assert list(z_["genes"]) == SP_GENES, "spatial cache was built for a different gene list — delete it"
else:
    Xg = g["X"]; indptr = np.asarray(Xg["indptr"][:]); NNZ = int(indptr[-1])
    gcol = sp_genes.get_indexer(SP_GENES); lut = -np.ones(len(sp_genes), int); lut[gcol] = np.arange(len(gcol))
    is_r01 = np.isin(sp_genes, R01); is_alt = np.isin(sp_genes, ALT)
    DET_N = np.zeros((len(labels), len(SP_GENES))); R01_SUM = np.zeros(len(so)); ALT_SUM = np.zeros(len(so))
    t0, CH = time.time(), 60_000_000
    for s in range(0, NNZ, CH):
        e = min(s + CH, NNZ)
        ic = np.asarray(Xg["indices"][s:e]); vals = np.asarray(Xg["data"][s:e])
        m = lut[ic] >= 0
        rows = np.searchsorted(indptr, s + np.nonzero(m)[0], side="right") - 1
        flat = lab_idx[rows] * len(SP_GENES) + lut[ic[m]]
        DET_N += np.bincount(flat, weights=(vals[m] > 0).astype(float),
                             minlength=DET_N.size).reshape(DET_N.shape)
        for flag, acc in [(is_r01, R01_SUM), (is_alt, ALT_SUM)]:
            mm = flag[ic]
            r2 = np.searchsorted(indptr, s + np.nonzero(mm)[0], side="right") - 1
            acc += np.bincount(r2, weights=vals[mm], minlength=len(so))
        print(f"  streaming spatial X: {e/NNZ:5.1%} {time.time()-t0:5.0f}s", end="\r", flush=True)
    print()
    np.savez_compressed(SP_CACHE, det=DET_N, r01=R01_SUM, alt=ALT_SUM, genes=np.array(SP_GENES))
n_lab = np.bincount(lab_idx, minlength=len(labels))
SPDET = pd.DataFrame(DET_N / n_lab[:, None], index=labels, columns=SP_GENES).T
SPDET.to_csv(OUT/"06_spatial_detection_by_label.csv")
REAL = [l for l in labels if l not in ("Unknown", "Erythrocyte", "Macrophage", "Monocyte", "Tcell", "DC")]
print(f"spatial: {len(so):,} cells, {len(SP_GENES)} genes, labels {REAL}")
print("\nspatial detection (% of cells) for marker genes, selected labels:")
print((100 * SPDET.loc[[x for x in ["Gzmb", "Prf1", "Nkg7", "Cd8a", "H2-Ab1", "Cd74", "Cxcl9", "Cxcl10",
                                    "Ccl5", "Cxcr3", "Spp1", "Cd44", "Fcgr4", "Srgn", "Lamp1", "Bnip3"] if x in SPDET.index],
                       ["CD8_T", "NK", "B", "M1_like_Mac", "M2_like_Mac", "Cancer_cell", "N2_like_Neu", "Unknown"]])
      .round(1).to_string())

# R01 vs alternative score composition of top scorers in spatial (per tissue top 15%, treated)
so["R01"] = R01_SUM / max(len([x for x in R01 if x in sp_genes]), 1)
so["ALT"] = ALT_SUM / max(len([x for x in ALT if x in sp_genes]), 1)
tr_sp = so[so.condition != "NoTx"]
tops = {}
for sc_ in ["R01", "ALT"]:
    q = tr_sp.groupby("tissue")[sc_].transform(lambda v: v.quantile(TOP_Q))
    t = tr_sp[tr_sp[sc_] >= q]
    tops[sc_] = (t.groupby("tissue").c2l_consolidated.value_counts(normalize=True).unstack().fillna(0).mean())
TOPS = pd.DataFrame(tops).sort_values("R01", ascending=False)
sp_lym = tr_sp.c2l_consolidated.isin(LYMPH_CT).values
for nm in ["R01", "ALT"]:
    DISC[("spatial", nm)] = discrim(tr_sp[nm].values, sp_lym)
    known = ~tr_sp.c2l_consolidated.isin(["Unknown", "Erythrocyte"]).values
    auc_k = roc_auc_score(sp_lym[known], tr_sp[nm].values[known])
    print(f"spatial {nm:<4s} AUROC lymphocyte label vs rest {DISC[('spatial', nm)][0]:.3f} (annotated cells only "
          f"{auc_k:.3f}) | precision at k={DISC[('spatial', nm)][2]:,}: {DISC[('spatial', nm)][1]:.2f} "
          f"(base rate {sp_lym.mean():.3f})")
print("\nSPATIAL — composition of the top 15% scorers per tissue (mean over treated tissues):")
print((100 * TOPS).round(1).head(12).to_string())


def transfer(gene, label):
    """measurable: detected in the intended label (>= 5%) and >= 1.5x B cells (the spillover control).
    label_specific: additionally >= 1.5x the median of the other annotated labels.
    Returns (detection in label, ratio to B, measurable, label_specific, highest-detecting label)."""
    if gene not in SPDET.index or label not in SPDET.columns: return np.nan, np.nan, False, False, ""
    own = SPDET.loc[gene, label]
    b = SPDET.loc[gene, "B"] if "B" in SPDET.columns else np.nan
    others = SPDET.loc[gene, [l for l in REAL if l != label]]
    meas = bool(own >= 0.05 and own >= 1.5 * max(b, 1e-6))
    spec_ = bool(meas and own >= 1.5 * max(others.median(), 1e-6))
    return own, own / max(b, 1e-6), meas, spec_, SPDET.loc[gene, REAL].idxmax()

spatial: 2,088,557 cells, 102 genes, labels ['B', 'CD4_T', 'CD8_T', 'Cancer_cell', 'Classical_Mono', 'Endothelial', 'Fibroblast', 'Intermediate_Mac', 'M1_like_Mac', 'M2_like_Mac', 'N1_like_Neu', 'N2_like_Neu', 'NK', 'NKT', 'Neutrophil', 'Nonclassical_Mono', 'Treg', 'cDC', 'pDC']

spatial detection (% of cells) for marker genes, selected labels:
        CD8_T    NK     B  M1_like_Mac  M2_like_Mac  Cancer_cell  N2_like_Neu  Unknown
Gzmb     14.3  27.6   6.8         26.8         15.4          4.2          4.7      8.9
Prf1      9.8  23.0   4.3         15.8         10.6          2.6          2.8      5.5
Nkg7      4.8  10.2   1.5          8.1          4.0          0.7          0.9      2.3
H2-Ab1   39.9  34.1  26.5         93.8         57.6         16.7         20.0     41.4
Cd74     55.8  51.4  42.5         98.8         81.9         30.2         31.8     60.2
Cxcl9    27.4  20.1  11.9         76.7         19.1          6.9          8.9     20.3
Cxcl10   22.1  14.3  11.4         39.3      

spatial R01  AUROC lymphocyte label vs rest 0.546 (annotated cells only 0.526) | precision at k=65,205: 0.11 (base rate 0.042)


spatial ALT  AUROC lymphocyte label vs rest 0.547 (annotated cells only 0.548) | precision at k=65,205: 0.12 (base rate 0.042)

SPATIAL — composition of the top 15% scorers per tissue (mean over treated tissues):
                   R01   ALT
c2l_consolidated            
Unknown           32.2  37.2
N1_like_Neu       12.3  10.8
Erythrocyte       11.5  10.3
Neutrophil         9.4  10.9
Classical_Mono     6.8   6.0
M2_like_Mac        6.7   4.9
Cancer_cell        5.1   4.6
N2_like_Neu        3.5   2.5
NK                 2.9   3.1
M1_like_Mac        2.5   2.3
CD8_T              2.1   2.2
Fibroblast         1.1   1.3


## 6. The shortlist — programs

In [6]:
TIER_PTS = {"Definitive": 3, "Supported": 2, "Suggestive": 1, "No evidence": 0}
cand = LED[LED.tier.isin(["Definitive", "Supported", "Suggestive"])].copy()
best = (cand.assign(pts=cand.tier.map(TIER_PTS))
            .sort_values("pts", ascending=False)
            .groupby(["program", "group"])
            .agg(best_tier=("tier", "first"), pts=("pts", "max"),
                 effects=("effect", lambda v: ", ".join(sorted(set(v)))),
                 direction=("score_main", lambda v: ", ".join(f"{x:+.1f}" for x in v)))
            .reset_index())
rows = []
for _, r in best.iterrows():
    lab = TO_SPATIAL.get(r.group)
    margin, share = specificity(r.program, r.group)
    genes_ = [x for x in PROGRAMS[r.program]["genes"] if x in SPDET.index]
    tr_ = [(x, *transfer(x, lab)) for x in genes_]
    measurable = [x for x, own, rb, ms, sp_, top_ in tr_ if ms]
    passing = [x for x, own, rb, ms, sp_, top_ in tr_ if sp_]
    elsewhere = [f"{x}>{top_}" for x, own, rb, ms, sp_, top_ in tr_ if top_ and top_ != lab and
                 SPDET.loc[x, top_] > 1.5 * max(own, 1e-6)]
    sx = SPPX[(SPPX.group == r.group) & (SPPX.program == r.program)]
    rows.append(dict(program=r.program, family=PROGRAMS[r.program]["family"], sc_group=r.group, spatial_label=lab,
                     best_tier=r.best_tier, effects=r.effects, direction=r.direction,
                     specificity_margin=margin, contribution_share=share,
                     spp1_cd44_partial_r=sx.partial_r.iloc[0] if len(sx) else np.nan,
                     spp1_cd44_p=sx.p_FL.iloc[0] if len(sx) else np.nan,
                     genes_in_spatial=len(genes_), genes_measurable=len(measurable), genes_label_specific=len(passing),
                     transferable_frac=len(measurable) / max(len(genes_), 1),
                     recommended_spatial_genes=", ".join((passing or measurable)[:12]),
                     higher_in_other_label=", ".join(elsewhere[:6]),
                     ambient_suspect=bool(LED[(LED.program == r.program) & (LED.group == r.group)].ambient_suspect.any())))
SH = pd.DataFrame(rows)
SH["transferable"] = SH.transferable_frac >= 0.5
SH["spatial_use"] = np.where(SH.ambient_suspect, "do not map (ambient)",
                    np.where(SH.genes_label_specific >= 0.5 * SH.genes_in_spatial, "cell-type readout",
                    np.where(SH.transferable, "tissue-level field (not label-specific)", "not measurable")))
SH["rank_score"] = (SH.best_tier.map(TIER_PTS) + SH.transferable.astype(int) + (SH.specificity_margin > 0).astype(int)
                    + (SH.spp1_cd44_p < 0.10).fillna(False).astype(int) - 2 * SH.ambient_suspect.astype(int))
SH = SH.sort_values(["rank_score", "best_tier"], ascending=[False, True])
SH.to_csv(OUT/"spatial_handoff_programs.csv", index=False)
print(SH[["rank_score", "program", "sc_group", "spatial_label", "best_tier", "effects", "direction",
          "specificity_margin", "spp1_cd44_partial_r", "spp1_cd44_p", "genes_measurable", "genes_label_specific",
          "genes_in_spatial", "spatial_use", "higher_in_other_label"]].round(2).to_string(index=False))

 rank_score                  program           sc_group    spatial_label  best_tier                effects        direction  specificity_margin  spp1_cd44_partial_r  spp1_cd44_p  genes_measurable  genes_label_specific  genes_in_spatial                             spatial_use                                                                                                            higher_in_other_label
          5            IFNG_Hallmark     Classical_Mono   Classical_Mono Definitive      PSCA, indirect RT       +2.9, -0.7                0.16                  NaN          NaN                25                    24                28                       cell-type readout                                                               Cd38>B, Ciita>M1_like_Mac, Irf4>Intermediate_Mac, Xcl1>M1_like_Mac
          5                 ISG_core     Classical_Mono   Classical_Mono Definitive      PSCA, indirect RT       +2.9, -1.5                0.29                  NaN          NaN             

## 7. The chemokine axis table, with spatial transferability

In [7]:
ax = AXT.copy()
ax["ligand_spatial_label"] = ax.source.map(TO_SPATIAL); ax["receptor_spatial_label"] = ax.target.map(TO_SPATIAL)
lt = [transfer(r.ligand, r.ligand_spatial_label) for _, r in ax.iterrows()]
rt = [transfer(r.receptor, r.receptor_spatial_label) for _, r in ax.iterrows()]
ax["ligand_spatial_det"] = [x[0] for x in lt]; ax["ligand_over_B"] = [x[1] for x in lt]
ax["ligand_transferable"] = [x[2] for x in lt]; ax["ligand_label_specific"] = [x[3] for x in lt]
ax["ligand_top_spatial_label"] = [x[4] for x in lt]
ax["receptor_spatial_det"] = [x[0] for x in rt]; ax["receptor_transferable"] = [x[2] for x in rt]
ax["in_liana_mouseconsensus"] = [f"{r.ligand}->{r.receptor}" not in missing_mc for _, r in ax.iterrows()]
cled = CLED[CLED.role == "ligand"].set_index(["gene", "group", "effect"])
ax["ligand_tier"] = [cled.observed_transcriptional_change.get((r.ligand, r.source, r.effect), "No evidence")
                     if (r.ligand, r.source, r.effect) in cled.index else "No evidence" for _, r in ax.iterrows()]
spp_l = SPPX.set_index(["group", "program"])
ax["spp1_cd44_relation"] = [
    (f"r={spp_l.loc[(r.source, f'{r.axis}_ligands'), 'partial_r']:+.2f}, p={spp_l.loc[(r.source, f'{r.axis}_ligands'), 'p_FL']:.3f}"
     if (r.source, f"{r.axis}_ligands") in spp_l.index else "") for _, r in ax.iterrows()]
cols = ["axis", "priority", "ligand", "source", "receptor", "target", "effect", "ligand_tier", "ligand_lfc",
        "ligand_best_q", "ligand_CARxRT_lfc", "ligand_CARxRT_q", "receptor_pct_pos", "receptor_measurable",
        "population_partial_r", "population_q", "within_mouse_r", "IFN_link_within_rho", "IFN_link_across_r",
        "cytotoxic_link", "inferred_chemotactic_potential", "spp1_cd44_relation", "ligand_spatial_label",
        "ligand_spatial_det", "ligand_over_B", "ligand_transferable", "ligand_label_specific",
        "ligand_top_spatial_label", "receptor_spatial_label",
        "receptor_spatial_det", "receptor_transferable", "in_liana_mouseconsensus"]
CAX = ax[cols]
CAX.to_csv(OUT/"chemokine_axis_table.csv", index=False)
pri = CAX[(CAX.ligand_tier.isin(["Definitive", "Supported"])) | (CAX.population_q < 0.10)]
print(f"axis rows with a Definitive/Supported ligand change or a population association: {len(pri)}")
print(pri[["axis", "ligand", "source", "receptor", "target", "effect", "ligand_tier", "ligand_lfc",
           "population_partial_r", "population_q", "IFN_link_within_rho", "ligand_transferable",
           "receptor_transferable", "in_liana_mouseconsensus"]].round(2).to_string(index=False))

axis rows with a Definitive/Supported ligand change or a population association: 49
 axis ligand             source receptor      target      effect ligand_tier  ligand_lfc  population_partial_r  population_q  IFN_link_within_rho  ligand_transferable  receptor_transferable  in_liana_mouseconsensus
CXCR3  Cxcl9 IFN_stimulated_Mac    Cxcr3       CD8_T        PSCA  Definitive +3.10/+2.85                  0.07          0.99                 0.22                 True                  False                    False
CXCR3  Cxcl9 IFN_stimulated_Mac    Cxcr3       CD8_T indirect RT  Definitive -0.77/-1.23                  0.07          0.99                 0.22                 True                  False                    False
CXCR3  Cxcl9 IFN_stimulated_Mac    Cxcr3          NK        PSCA  Definitive +3.10/+2.85                  0.08          0.99                 0.22                 True                  False                    False
CXCR3  Cxcl9 IFN_stimulated_Mac    Cxcr3          NK ind

## 8. Do not map spatially

In [8]:
dnm = []
if comp_top.reindex(["CD8_T", "NK", "NKT", "CD4_T", "Treg"]).fillna(0).sum() < 0.5:
    dnm.append(("R01 granzyme score (GOBP_GRANZYME_MEDIATED...)",
                f"top-15% scorers in single cell are {100*comp_top.reindex(['CD8_T','NK','NKT','CD4_T','Treg']).fillna(0).sum():.0f}% "
                f"lymphocytes; driven by {', '.join(x for x in ['Fcgr4','Srgn','Lamp1','Bnip3'] if x in R01)} in myeloid cells"))
for _, r in SH[SH.ambient_suspect].iterrows():
    dnm.append((f"{r.program} in {r.sc_group}", "ambient-suspect (macrophage RNA in a non-macrophage compartment)"))
for _, r in SH[(SH.spatial_use == "not measurable") & SH.best_tier.isin(["Definitive", "Supported"])].iterrows():
    dnm.append((f"{r.program} in {r.spatial_label}", f"only {r.genes_measurable}/{r.genes_in_spatial} genes "
                f"measurable above the B-cell spillover floor in that label"))
for gname in ["Gzmb", "Prf1", "Nkg7", "Gzma"]:
    if gname in SPDET.index:
        top_ = SPDET.loc[gname, REAL].idxmax()
        if top_ not in ("CD8_T", "NK"):
            dnm.append((f"{gname} as a lymphocyte marker in spatial",
                        f"detected most in '{top_}' ({100*SPDET.loc[gname, top_]:.0f}%) vs CD8_T "
                        f"{100*SPDET.loc[gname, 'CD8_T']:.0f}% / NK {100*SPDET.loc[gname, 'NK']:.0f}%"))
for gname, labs in [("Cxcr3", ["CD8_T", "NK", "Treg"]), ("Xcr1", ["cDC"]), ("Ccr4", ["Treg", "NKT", "CD4_T"]),
                    ("Cxcr6", ["CD8_T", "NKT", "CD4_T"]), ("Ccr5", ["CD8_T", "NK"])]:
    for lab in labs:
        own, rb, ok, _, _ = transfer(gname, lab)
        if pd.notna(own) and not ok:
            dnm.append((f"receptor {gname} as a per-cell readout in {lab}",
                        f"spatial detection {100*own:.1f}%, {rb:.1f}x B cells — not distinguishable from spillover"))
DNM = pd.DataFrame(dnm, columns=["item", "reason"]).drop_duplicates("item")
DNM.to_csv(OUT/"spatial_do_not_map.csv", index=False)
print(DNM.to_string(index=False))

                                          item                                                                                            reason
R01 granzyme score (GOBP_GRANZYME_MEDIATED...) top-15% scorers in single cell are 10% lymphocytes; driven by Srgn, Lamp1, Bnip3 in myeloid cells
                        cyto_effector in CD8_T                         only 4/11 genes measurable above the B-cell spillover floor in that label
             exh_inhibitory_receptors in CD8_T                          only 2/7 genes measurable above the B-cell spillover floor in that label
                  ISG_core in Intermediate_Mac                         only 4/10 genes measurable above the B-cell spillover floor in that label
                             ISG_core in CD8_T                         only 3/10 genes measurable above the B-cell spillover floor in that label
                 cyto_effector_noCcl5 in CD8_T                         only 3/10 genes measurable above the B-cell spillover floor

## 9. Summary

In [9]:
print("=" * 100); print("SUMMARY — report these back"); print("=" * 100)
print(f"\n[R01 cytotoxicity score] top-15% scorers in single cell: "
      + ", ".join(f"{k} {100*v:.0f}%" for k, v in comp_top.head(5).items()))
print(f"    in spatial: " + ", ".join(f"{k} {100*v:.0f}%" for k, v in TOPS.R01.head(5).items()))
print(f"    lymphocyte-restricted alternative {ALT}: top-15% in spatial "
      + ", ".join(f"{k} {100*v:.0f}%" for k, v in TOPS.ALT.sort_values(ascending=False).head(5).items()))
print("    discrimination of lymphocytes (AUROC | precision at k = #lymphocytes):")
for (mod, nm), (auc, prec, k) in DISC.items():
    print(f"      {mod:<11s} {nm:<10s} AUROC {auc:.3f} | precision {prec:.2f}")
print("\n[SPP1 -> CD44 potential (single-cell LIANA) vs programs, treated, car/rt/site partialled]")
for _, r in SPPX.sort_values("p_FL").iterrows():
    print(f"    receiver {r.spp1_cd44_receiver:<14s} {r.program:<26s} in {r.group:<18s} partial r {r.partial_r:+.2f} "
          f"p_FL {r.p_FL:.3f} q {r.q_BH:.2f} (n={r.n})")
print(f"\n[LIANA mouseconsensus is missing] {missing_mc}")
print("\n[top of the program shortlist]")
for _, r in SH.head(12).iterrows():
    print(f"    {r.rank_score:>2.0f}  {r.program:<26s} {r.sc_group:<18s} -> {r.spatial_label:<15s} "
          f"{r.best_tier:<11s} [{r.effects}] {r.direction}  | {r.spatial_use}; genes: {r.recommended_spatial_genes}")
print("\n[chemokine axes worth mapping — Definitive/Supported ligand change, transferable]")
cx = CAX[CAX.ligand_tier.isin(["Definitive", "Supported"]) & CAX.ligand_transferable]
for (axn, lig, src), d in cx.groupby(["axis", "ligand", "source"]):
    print(f"    {axn:<6s} {lig:<7s} from {src:<18s} effects {sorted(set(d.effect))}  receptor targets "
          f"{sorted(set(d.target[d.receptor_transferable]))} (transferable)  liana default: "
          f"{'yes' if d.in_liana_mouseconsensus.iloc[0] else 'NO — add the pair'}")
print("\n[do not map]")
for _, r in DNM.iterrows(): print(f"    {r['item']}: {r.reason}")
print(f"\n[files] {OUT}: spatial_handoff_programs.csv, chemokine_axis_table.csv, spatial_do_not_map.csv,"
      f"\n        06_spp1_cd44_vs_programs.csv, 06_sc_liana_per_sample.csv, 06_spatial_detection_by_label.csv")
print("=" * 100)

SUMMARY — report these back

[R01 cytotoxicity score] top-15% scorers in single cell: M2_like_Mac 45%, Classical_Mono 23%, M1_like_Mac 11%, NK 7%, Intermediate_Mac 5%
    in spatial: Unknown 32%, N1_like_Neu 12%, Erythrocyte 12%, Neutrophil 9%, Classical_Mono 7%
    lymphocyte-restricted alternative ['Gzma', 'Gzmb', 'Gzmc', 'Nkg7', 'Prf1', 'Ifng', 'Fasl', 'Ctsw', 'Klrd1']: top-15% in spatial Unknown 37%, Neutrophil 11%, N1_like_Neu 11%, Erythrocyte 10%, Classical_Mono 6%
    discrimination of lymphocytes (AUROC | precision at k = #lymphocytes):
      single cell R01_score  AUROC 0.822 | precision 0.57
      single cell ALT_score  AUROC 0.953 | precision 0.83
      spatial     R01        AUROC 0.546 | precision 0.11
      spatial     ALT        AUROC 0.547 | precision 0.12

[SPP1 -> CD44 potential (single-cell LIANA) vs programs, treated, car/rt/site partialled]
    receiver CD8_T          exh_inhibitory_receptors   in CD8_T              partial r -0.67 p_FL 0.011 q 0.10 (n=16)
    rece